# contiguous-layout — worked example 2: Fix a reshape-after-permute error with .contiguous()

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `contiguous-layout`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`permute` (like `transpose`) returns a **non-contiguous view** — it only relabels strides, it does not move data. `view` refuses non-contiguous inputs because it needs the requested shape to be expressible with a single stride pattern over the existing buffer. Inserting `.contiguous()` materializes a fresh row-major copy so the subsequent `view` succeeds.

## Worked solution

We have `x` of shape `(B, C, H, W)` and want to bring the spatial axes together as `(B, H, W, C)` and then flatten the trailing two axes to get `(B, H, W*C)`.

**Step 1 — reorder the axes.** `x.permute(0, 2, 3, 1)` puts the tensor in logical `(B, H, W, C)` order. But this is just a view with shuffled strides — the bytes in memory are still in the original `(B, C, H, W)` order, so the result is *not* contiguous.

**Step 2 — why a naive `.view()` fails.** `view(B, H, W*C)` wants to fuse the last two logical axes into one contiguous run of `W*C` elements. After the permute those elements are *not* adjacent in memory, so PyTorch raises `RuntimeError: view size is not compatible with input tensor's size and stride`.

**Step 3 — materialize first.** `.contiguous()` copies the data into a fresh buffer laid out row-major for the *current* logical shape `(B, H, W, C)`. Now the last two axes really are adjacent, so `.view(B, H, W*C)` is legal and cheap.

**Step 4 — confirm.** The result equals `x.permute(0,2,3,1).reshape(B, H, W*C)` (`reshape` is exactly `contiguous().view()` when a copy is needed), and `is_contiguous()` is `True`.

In [ ]:
def permute_then_flatten(x):
    B, C, H, W = x.shape
    return x.permute(0, 2, 3, 1).contiguous().view(B, H, W * C)

t.manual_seed(0)
x = t.arange(2 * 3 * 4 * 5).reshape(2, 3, 4, 5).float()
out = permute_then_flatten(x)
ref = x.permute(0, 2, 3, 1).reshape(2, 4, 5 * 3)
print("shape:", tuple(out.shape))
print("contiguous:", out.is_contiguous())
print("matches reshape:", t.equal(out, ref))